In [4]:
import serial
import struct
import time
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------
SERIAL_PORT = '/dev/tty.usbmodem0010500109063' 
BAUD_RATE = 115200
IMU_COLS = [
    'aX-f', 'aY-f', 'az-f', 'gX-f', 'gY-f', 'gZ-f', 
    'aX-c', 'aY-c', 'az-c', 'gX-c', 'gY-c', 'gZ-c', 
    'aX-q', 'aY-q', 'az-q', 'gX-q', 'gY-q', 'gZ-q', 
    'aX-h', 'aY-h', 'az-h', 'gX-h', 'gY-h', 'gZ-h'
]

# ---------------------------------------------------------
# 2. STREAMING FUNCTION
# ---------------------------------------------------------
def stream_data_to_board():
    print("Loading Dataset...")
    data_dir = Path("/Users/li0o0sun/projects/Projectwork2026/data")
    file_path = data_dir / "09_filtered_all_data.pkl"
    df = pd.read_pickle(file_path)
    
    # Filter out weighted variants and grab just one specific activity/rep to test
    df = df[~df['Activity'].str.endswith('_')]
    
    # Isolate just "walk" and Repetition 1 for a clean test stream
    df_test = df[(df['Activity'] == 'walk') & (df['Reps'] == 1)].copy()
    print(f"Found {len(df_test)} rows for Walk Rep 1. Starting stream...")

    print(f"Opening connection on {SERIAL_PORT}...")
    try:
        ser = serial.Serial()
        ser.port = SERIAL_PORT
        ser.baudrate = BAUD_RATE
        ser.timeout = 0  # NON-BLOCKING! Crucial for continuous streaming
        ser.setDTR(False)
        ser.open()
        
        ser.reset_input_buffer()
        ser.reset_output_buffer()
        time.sleep(2) # Wait for board to boot
        
        if ser.in_waiting > 0:
            print(f"Board Boot Message: {ser.read(ser.in_waiting).decode('utf-8', errors='ignore').strip()}")

        rows_sent = 0
        
        # ---------------------------------------------------------
        # 3. CONTINUOUS STREAM LOOP
        # ---------------------------------------------------------
        for index, row in df_test.iterrows():
            
            # Extract the 24 IMU values into a list
            imu_data = row[IMU_COLS].tolist()
            
            # Pack the 24 floats into a 96-byte binary packet ('<24f')
            binary_packet = struct.pack('<24f', *imu_data)
            ser.write(binary_packet)
            rows_sent += 1
            
            # CHECK FOR BOARD RESPONSE (Prediction)
            # We don't block. We just check if the board sent something back.
            if ser.in_waiting > 0:
                # Read all available lines
                lines = ser.readlines()
                for line in lines:
                    msg = line.decode('utf-8', errors='ignore').strip()
                    if msg:
                        print(f"[{rows_sent} rows sent] Board Reply: {msg}")
            
            # SIMULATE REAL-WORLD SENSOR TIMING (Crucial!)
            # If we blast data at full USB speed, the nRF5340 UART buffer will overflow.
            # Assuming data was recorded at 100Hz, we sleep for 0.01 seconds (10ms) per row.
            time.sleep(0.01) 
            
        print("\n Finished streaming Repetition 1!")
        ser.close()
        
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    stream_data_to_board()

Loading Dataset...
Found 2005 rows for Walk Rep 1. Starting stream...
Opening connection on /dev/tty.usbmodem0010500109063...
[201 rows sent] Board Reply: Pred Angle: -12848
[301 rows sent] Board Reply: Pred Angle: -12440
[401 rows sent] Board Reply: Pred Angle: -11767
[502 rows sent] Board Reply: Pred Angle: -11555
[601 rows sent] Board Reply: Pred A
[602 rows sent] Board Reply: ngle: -11283
[701 rows sent] Board Reply: Pred Angle: -12122
[801 rows sent] Board Reply: Pred Angle: -12182
[902 rows sent] Board Reply: Pred Angle: -11578
[1001 rows sent] Board Reply: Pred Angle: -10990
[1101 rows sent] Board Reply: Pred A
[1102 rows sent] Board Reply: ngle: -9416
[1201 rows sent] Board Reply: Pre
[1202 rows sent] Board Reply: d Angle: -8712
[1301 rows sent] Board Reply: P
[1302 rows sent] Board Reply: red Angle: -9540
[1401 rows sent] Board Reply: Pr
[1402 rows sent] Board Reply: ed Angle: -11012
[1501 rows sent] Board Reply: Pred Angle: -11213
[1601 rows sent] Board Reply: Pred Angle: -10